<a href="https://colab.research.google.com/github/PavaniMaganti/openai-competitor-demand-prediction/blob/main/Competitor_Demand_Prediction_Using_Job_Postings_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI Job Posting Data Collection and Validation

## Project Overview

This notebook implements a data collection and validation pipeline for the CadetX project, *Competitor Demand Prediction Using Job Postings (NLP & Forecasting)*.

The objective is to collect job-posting records for OpenAI from a multi-company job-postings dataset hosted on Hugging Face, inspect the available data, identify OpenAI records, and prepare a raw dataset for subsequent NLP and hiring-demand analysis.

## Methodology

The workflow consists of the following stages:

1. **Environment Setup:** Install and import the Python libraries required for data processing and Parquet file handling.
2. **Company Data Inspection:** Load the company lookup table and identify OpenAI's company record and unique company ID.
3. **Dataset Discovery:** Access the Hugging Face dataset repository and identify the available job-posting data files.
4. **Schema Inspection:** Load one job-data shard to examine its structure, columns and sample records.
5. **Company Identification:** Match the OpenAI company ID against job records to identify relevant postings.
6. **Data Collection:** Read the 64 available job-data shards using streaming and retain records belonging to OpenAI.
7. **Data Validation:** Check record counts, missing descriptions, posting dates, duplicate identifiers, application URLs, locations, departments and workplace information.
8. **Lifecycle Inspection:** Examine posting and lifecycle identifiers to understand the dataset's record structure.
9. **Data Export:** Convert the collected OpenAI records into a Pandas DataFrame and export the raw dataset as a CSV file.

## Initial Results

- OpenAI job postings identified: **863**
- Non-empty job descriptions: **863**
- Missing posting dates: **0**
- Duplicate posting IDs: **0**
- Duplicate lifecycle IDs: **0**
- Missing application URLs: **0**
- Historical posting date range: **25 May 2023 – 24 November 2026**

## Final Output

The pipeline produces `openai_raw_dataset.csv`, containing the collected OpenAI job-posting records and their available attributes.

This raw dataset will serve as the input for subsequent data cleaning, NLP-based skill extraction, feature engineering, hiring trend analysis and demand forecasting.

## Important Data Limitations

City, region and location-country-code fields were missing in the inspected OpenAI records. However, the separate country field contained country-level information. The dataset's collection methodology, source permissions and licence conditions must also be documented and checked before final use.


In [1]:
!pip install -q pandas pyarrow

In [2]:

import pandas as pd

url = "https://huggingface.co/datasets/scholarweave/Jobs-Dataset-v2/resolve/main/data/companies/snapshot_date%3D2026-09-26/part-00000.parquet"

companies = pd.read_parquet(url)

print("Columns:")
print(companies.columns.tolist())

print("\nFirst five rows:")
display(companies.head())


Columns:
['id', 'name', 'website', 'ats', 'slug', 'unique_id', 'career_url', 'founded', 'size', 'locality', 'region', 'country', 'industry', 'linkedin_url', 'linkedin_id']

First five rows:


,id,name,website,ats,slug,unique_id,career_url,founded,size,locality,region,country,industry,linkedin_url,linkedin_id
0,2,ADA Bureau of Investigations & Security,adavt.com,adp,33574b24-a1c3-467a-bbff-bb9062661715,adp:33574b24-a1c3-467a-bbff-bb9062661715:wfn,https://workforcenow.adp.com/mascsr/default/md...,1996.0,201-500,colchester,vermont,united states,construction,linkedin.com/company/ada-traffic-control-ltd,raOjxcrBW9Yl8Xax06V8Rgp6xHp2
1,3,ADBRO,adbro.me,breezy,adbro,breezy:adbro,https://adbro.breezy.hr,NaN,1-10,leeds,england,united kingdom,furniture,linkedin.com/company/adbro-limited,etYAbv2ZXRh8bCTSGnHXdQ27yxfW
2,4,ADC Energy,adc.energy,smartrecruiters,ADCEnergy,smartrecruiters:ADCEnergy,https://jobs.smartrecruiters.com/ADCEnergy,1986.0,11-50,aberdeen,scotland,united kingdom,oil & energy,linkedin.com/company/adcriginspection,CEot0wCr3wiaFScQZOVbpQ6log4D
3,5,ADC Aerospace,adc-aerospace.com,paycom,623B0EE55D9C72D7ED827F8260F0B59C,paycom:623B0EE55D9C72D7ED827F8260F0B59C,https://www.paycomonline.net/v4/ats/web.php/po...,1946.0,51-200,buena park,california,united states,aviation & aerospace,linkedin.com/company/adc-aerospace,0KHcGuW3KlN7jNMvnjSroAGiwwyB
4,6,ADCCO,adccotn.com,bamboohr,adccontractors,bamboohr:adccontractors,https://adccontractors.bamboohr.com/careers,1984.0,1-10,lebanon,tennessee,united states,construction,linkedin.com/company/adccotn,WSGMzACXs8DMuT8xbMlrHw8C33x8


In [3]:
companies[companies["name"].str.contains("OpenAI", case=False, na=False)]

,id,name,website,ats,slug,unique_id,career_url,founded,size,locality,region,country,industry,linkedin_url,linkedin_id
71587,79598,OpenAI,openai.com,ashbyhq,openai,ashbyhq:openai,https://jobs.ashbyhq.com/openai,2015.0,201-500,san francisco,california,united states,research,linkedin.com/company/openai,30x8wUj6MNfspLBBgNqOawgOLDP9


In [6]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="scholarweave/Jobs-Dataset-v2",
    repo_type="dataset"
)

jobs_files = [
    f for f in files
    if f.startswith("data/active/minimal/")
    and f.endswith(".parquet")
]

print("Number of job files:", len(jobs_files))
print(*jobs_files[:10], sep="\n")

Number of job files: 64
data/active/minimal/current/shard_0000.parquet
data/active/minimal/current/shard_0001.parquet
data/active/minimal/current/shard_0002.parquet
data/active/minimal/current/shard_0003.parquet
data/active/minimal/current/shard_0004.parquet
data/active/minimal/current/shard_0005.parquet
data/active/minimal/current/shard_0006.parquet
data/active/minimal/current/shard_0007.parquet
data/active/minimal/current/shard_0008.parquet
data/active/minimal/current/shard_0009.parquet


In [7]:
import pandas as pd
from huggingface_hub import hf_hub_url

file_url = hf_hub_url(
    repo_id="scholarweave/Jobs-Dataset-v2",
    filename="data/active/minimal/current/shard_0000.parquet",
    repo_type="dataset"
)

jobs_sample = pd.read_parquet(file_url)

print("Rows:", len(jobs_sample))
print("\nColumns:")
print(jobs_sample.columns.tolist())

print("\nFirst 5 records:")
display(jobs_sample.head())

Rows: 25597

Columns:
['posting_id', 'lifecycle_id', 'lifecycle_sequence', 'company_id', 'title', 'department', 'employment_type', 'workplace_type', 'country', 'is_remote', 'posted_at', 'apply_url', 'first_seen_at', 'last_seen_at', 'description_plain', 'description_summary', 'location_city', 'location_region', 'location_country_code', 'salary_min', 'salary_max', 'salary_currency', 'salary_frequency', 'skills', 'content_hash', 'schema_version']

First 5 records:


,posting_id,lifecycle_id,lifecycle_sequence,company_id,title,department,employment_type,workplace_type,country,is_remote,...,location_city,location_region,location_country_code,salary_min,salary_max,salary_currency,salary_frequency,skills,content_hash,schema_version
0,adp:0060b91b-0b3b-468f-87c6-cc32e722a067:wfn/9...,adp:0060b91b-0b3b-468f-87c6-cc32e722a067:wfn/9...,1,64835,Discharge Planner,None,Full Time,tbc,United States,False,...,None,None,None,None,None,None,None,[],4bb7ce20d5e7a4abec50aa6f946138e2c81dde11fa8054...,2.0.0
1,adp:007c6afc-b329-461d-a98d-d324ea14ba49:wfn/9...,adp:007c6afc-b329-461d-a98d-d324ea14ba49:wfn/9...,1,83538,Business Developer,None,FullTime Exempt,tbc,United States,False,...,None,None,None,None,None,None,None,[],416405766e07115a5e785e7a31fba74c469e4ee6db5cc5...,2.0.0
2,adp:00a03d16-f196-4d63-8f72-1edbb765ffc7:wfn/9...,adp:00a03d16-f196-4d63-8f72-1edbb765ffc7:wfn/9...,1,17943,Early Head Start Teacher I-P (Represented),None,Full-time Position,tbc,United States,False,...,None,None,None,None,None,None,None,[],33665153a74185b82defc4d64b95ca057db37c146c12c5...,2.0.0
3,adp:00a03d16-f196-4d63-8f72-1edbb765ffc7:wfn/9...,adp:00a03d16-f196-4d63-8f72-1edbb765ffc7:wfn/9...,1,17943,Head Start Teacher II-B (Represented),None,Full-time Position,tbc,United States,False,...,None,None,None,None,None,None,None,[],305545789c3a41f3cb845830c4e716ae8ad9c436c4e6c5...,2.0.0
4,adp:00a5c9c8-8a84-48e7-be32-9c1ebfa8029b:wfn/9...,adp:00a5c9c8-8a84-48e7-be32-9c1ebfa8029b:wfn/9...,1,28779,Press Operator,None,None,tbc,United States,False,...,None,None,None,None,None,None,None,[],e1cc9a0762dbf2cced4d9f776892d603b37faf0ee7c9f6...,2.0.0


In [8]:
openai_jobs_sample = jobs_sample[
    jobs_sample["company_id"] == 71587
].copy()

print("OpenAI postings in this shard:", len(openai_jobs_sample))

print("\nRecords with descriptions:",
      openai_jobs_sample["description_plain"].notna().sum())

display(
    openai_jobs_sample[
        ["title", "posted_at", "description_plain"]
    ].head(3)
)

OpenAI postings in this shard: 0

Records with descriptions: 0


,title,posted_at,description_plain


In [9]:
companies[
    companies["name"].str.contains("OpenAI", case=False, na=False)
][["id", "name", "website"]]

,id,name,website
71587,79598,OpenAI,openai.com


In [10]:
openai_jobs_sample = jobs_sample[
    jobs_sample["company_id"] == 79598
].copy()

print("OpenAI postings:", len(openai_jobs_sample))

display(
    openai_jobs_sample[
        ["title", "posted_at", "description_plain"]
    ].head(3)
)

OpenAI postings: 8


,title,posted_at,description_plain
2617,"Growth Lead, Japan",2026-09-30 03:46:38.119000+00:00,About the Team\n\nThe Growth team drives user ...
2618,"Founding Software Engineer, Physical Engineering",2026-10-26 22:18:02.107000+00:00,ABOUT THE TEAM\n\nThe Physical Engineering tea...
2619,"Sales Manager, Global Ads Solutions - Retail &...",2026-11-17 01:42:03.813000+00:00,About the team\n\nThe Global Ad Solutions team...


In [11]:
from datasets import load_dataset
from huggingface_hub import hf_hub_url

# Create download links for all 64 job files
urls = [
    hf_hub_url(
        repo_id="scholarweave/Jobs-Dataset-v2",
        filename=file,
        repo_type="dataset"
    )
    for file in jobs_files
]

# Read the dataset in streaming mode
stream = load_dataset(
    "parquet",
    data_files={"train": urls},
    split="train",
    streaming=True
)

# Keep only OpenAI records
openai_records = []

for row in stream:
    if row["company_id"] == 79598:
        openai_records.append(row)

print("Total OpenAI postings:", len(openai_records))

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Total OpenAI postings: 863


In [12]:
import pandas as pd

openai_df = pd.DataFrame(openai_records)

print("Total records:", len(openai_df))

print(
    "Non-empty descriptions:",
    openai_df["description_plain"].notna().sum()
)

print(
    "Missing posting dates:",
    openai_df["posted_at"].isna().sum()
)

print(
    "Duplicate posting IDs:",
    openai_df["posting_id"].duplicated().sum()
)

Total records: 863
Non-empty descriptions: 863
Missing posting dates: 0
Duplicate posting IDs: 0


In [13]:
display(
    openai_df[
        ["title", "posted_at", "location_city"]
    ].head(10)
)

,title,posted_at,location_city
0,"Growth Lead, Japan",2026-09-30 03:46:38.119000+00:00,None
1,"Founding Software Engineer, Physical Engineering",2026-10-26 22:18:02.107000+00:00,None
2,"Sales Manager, Global Ads Solutions - Retail &...",2026-11-17 01:42:03.813000+00:00,None
3,"Senior Counsel, AI Product (Enterprise/API)",2026-11-05 21:11:34.409000+00:00,None
4,"Client Partner, New Business - Ads Solutions",2026-11-19 17:56:47.704000+00:00,None
5,Value Engineer - San Francisco,2026-10-26 21:21:48.233000+00:00,None
6,"B2B Partnerships Lead, Business Development",2026-11-24 02:46:12.799000+00:00,None
7,"Software Engineer, Data Infrastructure",2024-06-27 22:14:24.195000+00:00,None
8,"Software Engineer, Security Observability",2026-04-15 16:54:42.457000+00:00,None
9,"Data Scientist, Real Estate & Workplace",2026-08-20 16:38:56.595000+00:00,None


In [14]:
print("Empty descriptions:",
      openai_df["description_plain"].fillna("").str.strip().eq("").sum())

print("\nLocation information:")
print(openai_df[
    ["location_city", "location_region", "location_country_code"]
].notna().sum())

print("\nMissing application URLs:",
      openai_df["apply_url"].isna().sum())

print("\nPosting date range:")
print("Earliest:", openai_df["posted_at"].min())
print("Latest:", openai_df["posted_at"].max())

print("\nPosting date sample:")
display(openai_df[[
    "title", "posted_at", "location_city",
    "location_region", "location_country_code"
]].head(15))

Empty descriptions: 0

Location information:
location_city            0
location_region          0
location_country_code    0
dtype: int64

Missing application URLs: 0

Posting date range:
Earliest: 2023-05-25 04:06:29.356000+00:00
Latest: 2026-11-24 06:31:52.965000+00:00

Posting date sample:


,title,posted_at,location_city,location_region,location_country_code
0,"Growth Lead, Japan",2026-09-30 03:46:38.119000+00:00,None,None,None
1,"Founding Software Engineer, Physical Engineering",2026-10-26 22:18:02.107000+00:00,None,None,None
2,"Sales Manager, Global Ads Solutions - Retail &...",2026-11-17 01:42:03.813000+00:00,None,None,None
3,"Senior Counsel, AI Product (Enterprise/API)",2026-11-05 21:11:34.409000+00:00,None,None,None
4,"Client Partner, New Business - Ads Solutions",2026-11-19 17:56:47.704000+00:00,None,None,None
5,Value Engineer - San Francisco,2026-10-26 21:21:48.233000+00:00,None,None,None
6,"B2B Partnerships Lead, Business Development",2026-11-24 02:46:12.799000+00:00,None,None,None
7,"Software Engineer, Data Infrastructure",2024-06-27 22:14:24.195000+00:00,None,None,None
8,"Software Engineer, Security Observability",2026-04-15 16:54:42.457000+00:00,None,None,None
9,"Data Scientist, Real Estate & Workplace",2026-08-20 16:38:56.595000+00:00,None,None,None


In [15]:
print("Country values:")
print(openai_df["country"].value_counts(dropna=False).head(15))

print("\nWorkplace type:")
print(openai_df["workplace_type"].value_counts(dropna=False))

print("\nRemote status:")
print(openai_df["is_remote"].value_counts(dropna=False))

print("\nDepartment:")
print(openai_df["department"].value_counts(dropna=False).head(15))

Country values:
country
United States     658
Singapore          43
United Kingdom     35
Japan              33
Ireland            19
Germany            15
India              12
Australia          11
South Korea        10
Brazil              8
France              7
Canada              5
Spain               2
Switzerland         2
Sweden              2
Name: count, dtype: int64

Workplace type:
workplace_type
hybrid    590
tbc       225
onsite     25
remote     23
Name: count, dtype: int64

Remote status:
is_remote
True     613
False    250
Name: count, dtype: int64

Department:
department
Go To Market                    212
Applied AI                      125
Scaling                          85
Research                         57
User Operations                  55
Marketing                        47
Forward Deployed Engineering     43
Security                         35
People                           24
Finance                          23
Consumer Devices                 19
Robotics

In [16]:
print("Unique posting IDs:", openai_df["posting_id"].nunique())

print("Unique lifecycle IDs:", openai_df["lifecycle_id"].nunique())

print("\nLifecycle sequence distribution:")
print(openai_df["lifecycle_sequence"].value_counts().sort_index())

print("\nRepeated lifecycle IDs:",
      openai_df["lifecycle_id"].duplicated().sum())

Unique posting IDs: 863
Unique lifecycle IDs: 863

Lifecycle sequence distribution:
lifecycle_sequence
1    797
2     66
Name: count, dtype: int64

Repeated lifecycle IDs: 0


In [17]:
openai_df.to_csv(
    "openai_raw_dataset.csv",
    index=False
)

print("Dataset saved successfully!")
print("Shape:", openai_df.shape)

Dataset saved successfully!
Shape: (863, 26)


In [18]:
from google.colab import files

files.download("openai_raw_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>